# ERIC — Module 6e: Fully Separated Notebook

## Spatially aligned image + raster fusion Transformer

This version separates the workflow into small, sequential code cells. Imports, configuration, data loading, validation, helper functions, model classes, model initialization, training, checkpoint restoration, inference, qualitative inspection, metrics, and exports are separated wherever they form independent steps.

### Execution notes
- Run cells from top to bottom in order.
- Do not skip class/function-definition cells that later cells depend on.
- The notebook preserves the original Module 6e architecture and dataset paths.
- Training cells can take a long time; if the trained checkpoint already exists, you may skip the training loop only if the matching model/configuration is already defined and the checkpoint is available.
- Evaluation dependencies are installed in a dedicated cell.
- Test metrics are for final reporting only; do not use them to tune the model.


# ERIC — Module 6e: Spatially Aligned Image + Raster Fusion Transformer

## Research objective
Evaluate whether combining remote-sensing image features with a spatially aligned semantic raster helps generate captions. This is the image + raster fusion Transformer arm of ERIC.

**Inputs:** RGB image (224×224) and a 7×7 raster of 49 semantic label IDs.  
**Output:** one generated caption compared with five reference captions.

### Experimental controls
- Use official RSICD train/validation/test splits.
- Build the caption vocabulary from training captions only.
- Select checkpoints using validation loss, not test metrics.
- Standardize metric implementations before comparing Modules 6a, 6b, and 6e.
- Raster labels are Qwen-generated pseudo-labels and are not all manually verified.

## 1. Architecture and method

### Spatial alignment
ViT-Base/32 produces 49 spatial patch tokens (7×7) for a 224×224 image. The semantic raster has 49 cells in the same row-major order, so each raster cell is fused with its corresponding image patch token.

### Fusion pathway
1. Encode the image with ImageNet-pretrained ViT-Base/32.
2. Embed each raster label and add row/column positional embeddings.
3. Concatenate each image patch feature (768-D) and raster embedding (512-D).
4. Project each pair to the decoder's 512-D feature space.
5. Project the CLS token to 512-D and prepend it to the 49 fused tokens.
6. Condition an autoregressive Transformer decoder on the resulting 50-token memory.

### Training and limitations
The decoder uses teacher forcing and token-level cross-entropy with padding ignored. The final two ViT blocks and final normalization are trainable. Because raster labels are pseudo-labels derived from the image, the experiment measures the usefulness of the current raster pipeline, not perfect semantic segmentation. Edge-efficiency claims must include raster-generation cost.

## 2. Environment, paths, and configuration

Imports libraries, defines Kaggle input/output paths, fixes the eight selected scenes, sets the 7×7 grid, and defines training hyperparameters. Run first.

### Code step 1: Environment, paths, and configuration

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import os
import json
import math
import random
import re
import time
import csv
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torchvision
import timm

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### Code step 2: Execute environment, paths, and configuration

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

### Code step 3: Environment, paths, and configuration

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
IMG_DIR = Path(
    "/kaggle/input/datasets/vedantbarhate007/"
    "rsicd-eric/RSICD_optimal-master/RSICD_images"
)

DATASET_JSON = Path(
    "/kaggle/input/datasets/vedantbarhate007/"
    "rsicd-eric/RSICD_optimal-master/dataset_rsicd.json"
)

CLASSES_DIR = Path(
    "/kaggle/input/datasets/vedantbarhate007/"
    "rsicd-eric/RSICD_optimal-master/txtclasses_rsicd"
)

LABELS_DIR = Path(
    "/kaggle/input/datasets/vedantbarhate007/eric-qwenlabels"
)

GLOBAL_VOCAB_PATH = Path(
    "/kaggle/input/datasets/vedantbarhate007/"
    "rsicd-raster-classes/raster_classes_dataset/global_vocab.json"
)

# ----------------------------
# Output directory
# ----------------------------
OUTPUT_DIR = Path("/kaggle/working/ERIC_Module6e")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Experimental configuration
# ----------------------------
SELECTED_SCENES = {
    "airport",
    "baseballfield",
    "beach",
    "mediumresidential",
    "pond",
    "river",
    "sparseresidential",
    "stadium",
}

IMAGE_SIZE = 224
PATCH_SIZE = 32
GRID_SIZE = 7
NUM_PATCHES = GRID_SIZE * GRID_SIZE  # 49

# Caption decoder: aligned with Module 6a
D_MODEL = 512
NUM_DECODER_LAYERS = 3

### Code step 4: Environment, paths, and configuration

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
NUM_HEADS = 8
DIM_FEEDFORWARD = 2048
DROPOUT = 0.1
MAX_CAPTION_LEN = 64

# Training configuration; we can tune using validation results
BATCH_SIZE = 32
NUM_WORKERS = 2
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
EARLY_STOPPING_PATIENCE = 5

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("Device:", device)

### Code step 5: Execute environment, paths, and configuration

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1e9,
            2
        ),
        "GB"
    )

### Code step 6: Environment, paths, and configuration

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
required_paths = {
    "Image directory": IMG_DIR,
    "RSICD metadata": DATASET_JSON,
    "Scene class directory": CLASSES_DIR,
    "Raster labels directory": LABELS_DIR,
    "Global raster vocabulary": GLOBAL_VOCAB_PATH,
}

### Code step 7: Execute environment, paths, and configuration

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for name, path in required_paths.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")
    assert path.exists(), f"{name} not found: {path}"

### Code step 8: Environment, paths, and configuration

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert torch.cuda.is_available(), (
    "Enable a Kaggle GPU before training Module 6e."
)

### Code step 9: Environment, paths, and configuration

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert IMAGE_SIZE // PATCH_SIZE == GRID_SIZE

### Code step 10: Environment, paths, and configuration

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nModule 6e configuration initialized.")
print("Selected scenes:", sorted(SELECTED_SCENES))
print("Image grid:", f"{GRID_SIZE}x{GRID_SIZE}")
print("Image patches:", NUM_PATCHES)
print("Output directory:", OUTPUT_DIR)

## 3. Load and validate raster vocabulary and labels

Loads the global vocabulary with stable IDs and validates each image's 49 row-major cell labels.

### Code step 11: Execute load and validate raster labels

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
with open(GLOBAL_VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab_data = json.load(f)

### Code step 12: Execute load and validate raster labels

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if isinstance(vocab_data, list):
    vocab_by_id = {
        int(item["index"]): str(item["term"]).strip()
        for item in vocab_data
    }
elif isinstance(vocab_data, dict):
    # Support a dictionary mapping IDs to terms if encountered.
    vocab_by_id = {
        int(k): str(v).strip()
        for k, v in vocab_data.items()
    }
else:
    raise TypeError(
        f"Unsupported global vocabulary format: {type(vocab_data)}"
    )

### Code step 13: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert vocab_by_id, "Global vocabulary is empty."

### Code step 14: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert sorted(vocab_by_id) == list(range(len(vocab_by_id))), (
    "Vocabulary indices must be contiguous and start at 0."
)

### Code step 15: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
term_to_id = {
    term: idx for idx, term in vocab_by_id.items()
}

### Code step 16: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(term_to_id) == len(vocab_by_id), (
    "Duplicate terms found in global vocabulary."
)

### Code step 17: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("Global vocabulary size:", len(vocab_by_id))
print("First 10 vocabulary entries:")

### Code step 18: Execute load and validate raster labels

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for idx in range(min(10, len(vocab_by_id))):
    print(f"  {idx}: {vocab_by_id[idx]}")

### Code step 19: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
SCENE_LABEL_FILES = {
    scene: LABELS_DIR / f"{scene}.json"
    for scene in sorted(SELECTED_SCENES)
}

missing_files = [
    str(path)
    for path in SCENE_LABEL_FILES.values()
    if not path.exists()
]

### Code step 20: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert not missing_files, (
    "Missing raster label files:\n" + "\n".join(missing_files)
)

### Code step 21: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
raster_records = {}
duplicate_filenames = []

### Code step 22: Execute load and validate raster labels

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for scene, path in SCENE_LABEL_FILES.items():
    with open(path, "r", encoding="utf-8") as f:
        scene_data = json.load(f)

    print(f"{scene}: {len(scene_data):,} images")

    for filename, entry in scene_data.items():
        if filename in raster_records:
            duplicate_filenames.append(filename)
            continue

        raster_records[filename] = {
            "scene": scene,
            "cells": entry.get("cells", []),
            "needs_review": bool(entry.get("needs_review", False)),
        }

### Code step 23: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert not duplicate_filenames, (
    "Duplicate filenames across scene label files: "
    + ", ".join(duplicate_filenames[:10])
)

### Code step 24: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nTotal raster records:", f"{len(raster_records):,}")

# ----------------------------
# Validate every raster record
# ----------------------------
invalid_cell_count = 0
invalid_cell_ids = 0
unknown_terms = Counter()
review_counts = Counter()
invalid_examples = []

expected_cell_ids = list(range(NUM_PATCHES))

### Code step 25: Execute load and validate raster labels

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for filename, record in raster_records.items():
    cells = record["cells"]
    review_counts[record["scene"]] += int(record["needs_review"])

    if len(cells) != NUM_PATCHES:
        invalid_cell_count += 1
        if len(invalid_examples) < 10:
            invalid_examples.append(
                (filename, f"Expected 49 cells, got {len(cells)}")
            )
        continue

    cell_ids = [cell.get("cell_id") for cell in cells]

    if cell_ids != expected_cell_ids:
        invalid_cell_ids += 1
        if len(invalid_examples) < 10:
            invalid_examples.append(
                (filename, "Cell IDs are missing, duplicated, or unordered")
            )

    for cell in cells:
        term = str(cell.get("label", "")).strip()

        if term not in term_to_id:
            unknown_terms[term] += 1

### Code step 26: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nValidation summary")
print("------------------")
print("Records with incorrect cell count:", invalid_cell_count)
print("Records with incorrect cell ordering:", invalid_cell_ids)
print("Unknown label occurrences:", sum(unknown_terms.values()))
print("Images flagged needs_review:", sum(review_counts.values()))

### Code step 27: Execute load and validate raster labels

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if unknown_terms:
    print("\nUnknown terms (up to 20):")
    for term, count in unknown_terms.most_common(20):
        print(f"  {term!r}: {count}")

### Code step 28: Execute load and validate raster labels

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if invalid_examples:
    print("\nInvalid examples:")
    for filename, reason in invalid_examples:
        print(f"  {filename}: {reason}")

### Code step 29: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert invalid_cell_count == 0, "Fix invalid raster cell counts first."

### Code step 30: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert invalid_cell_ids == 0, "Fix raster cell ordering first."

### Code step 31: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert not unknown_terms, (
    "Some raster labels are absent from global_vocab.json. "
    "Do not remap IDs silently; inspect the vocabulary and labels."
)

### Code step 32: Load and validate raster labels

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nRaster records passed structural validation.")
print("Vocabulary IDs have been preserved.")

## 4. Align raster records with RSICD metadata

Joins raster records with official image metadata, captions, scene names, image paths, and official splits. This prevents split leakage.

### Code step 33: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
with open(DATASET_JSON, "r", encoding="utf-8") as f:
    rsicd_data = json.load(f)

### Code step 34: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert "images" in rsicd_data, (
    "Expected dataset_rsicd.json to contain an 'images' list."
)

### Code step 35: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
metadata_by_filename = {}
metadata_duplicates = []

### Code step 36: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for item in rsicd_data["images"]:
    filename = item.get("filename")

    if not filename:
        continue

    if filename in metadata_by_filename:
        metadata_duplicates.append(filename)
        continue

    metadata_by_filename[filename] = item

### Code step 37: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert not metadata_duplicates, (
    "Duplicate filenames in RSICD metadata: "
    + ", ".join(metadata_duplicates[:10])
)

### Code step 38: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("RSICD metadata images:", len(metadata_by_filename))

### Code step 39: Define extract_captions

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def extract_captions(item):
    captions = []

    # Format used by the existing RSICD JSON:
    # item["sentences"] = [{"raw": "...", ...}, ...]
    sentences = item.get("sentences", [])

    if isinstance(sentences, list):
        for sentence in sentences:
            if isinstance(sentence, dict):
                caption = sentence.get("raw", "")
            elif isinstance(sentence, str):
                caption = sentence
            else:
                continue

            caption = str(caption).strip()
            if caption:
                captions.append(caption)

    # Some dataset variants store captions under "captions".
    if not captions:
        stored_captions = item.get("captions", [])

        if isinstance(stored_captions, list):
            for caption in stored_captions:
                if isinstance(caption, dict):
                    caption = caption.get("raw", "")

                caption = str(caption).strip()
                if caption:
                    captions.append(caption)

    return captions

### Code step 40: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
VALID_SPLITS = {"train", "val", "test"}

### Code step 41: Define normalize_split

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def normalize_split(value):
    value = str(value).strip().lower()

    # Keep only known official split names.
    if value in VALID_SPLITS:
        return value

    return None

### Code step 42: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
aligned_records = []
missing_metadata = []
missing_images = []
invalid_splits = []
scene_mismatches = []
missing_captions = []

### Code step 43: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for filename, raster in raster_records.items():
    metadata = metadata_by_filename.get(filename)

    if metadata is None:
        missing_metadata.append(filename)
        continue

    split = normalize_split(metadata.get("split"))

    if split is None:
        invalid_splits.append(
            (filename, metadata.get("split"))
        )
        continue

    # Infer scene from the raster file the record came from.
    raster_scene = raster["scene"]

    # The dataset's selected scene files use names such as
    # airport_49.jpg and mediumresidential_123.jpg.
    stem = Path(filename).stem
    inferred_scene = stem.rsplit("_", 1)[0].lower()

    if raster_scene not in SELECTED_SCENES:
        scene_mismatches.append(
            (filename, raster_scene, "unknown_scene")
        )
        continue

    image_path = IMG_DIR / filename

    if not image_path.exists():
        missing_images.append(filename)
        continue

    captions = extract_captions(metadata)

    if not captions:
        missing_captions.append(filename)
        continue

    # Preserve all supplied references, including duplicates,
    # to remain consistent with the dataset's existing captions.
    raster_ids = [
        term_to_id[str(cell["label"]).strip()]
        for cell in raster["cells"]
    ]

    aligned_records.append({
        "filename": filename,
        "scene": raster_scene,
        "split": split,
        "image_path": str(image_path),
        "raster_ids": raster_ids,
        "captions": captions,
        "needs_review": raster["needs_review"],
    })

### Code step 44: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nAlignment summary")
print("-----------------")
print("Aligned records:", len(aligned_records))
print("Missing metadata:", len(missing_metadata))
print("Missing image files:", len(missing_images))
print("Invalid or missing splits:", len(invalid_splits))
print("Scene mismatches:", len(scene_mismatches))
print("Records without captions:", len(missing_captions))

### Code step 45: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if missing_metadata:
    print("Missing metadata examples:", missing_metadata[:5])

### Code step 46: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if missing_images:
    print("Missing image examples:", missing_images[:5])

### Code step 47: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if invalid_splits:
    print("Invalid split examples:", invalid_splits[:5])

### Code step 48: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if scene_mismatches:
    print("Scene mismatch examples:", scene_mismatches[:5])

### Code step 49: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
if missing_captions:
    print("Caption-missing examples:", missing_captions[:5])

### Code step 50: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert not missing_metadata, "Some raster records have no RSICD metadata."

### Code step 51: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert not missing_images, "Some raster records have no corresponding image."

### Code step 52: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert not invalid_splits, "Some records have invalid official split names."

### Code step 53: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert not scene_mismatches, "Some raster filenames do not match their scene."

### Code step 54: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert not missing_captions, "Some records have no reference captions."

### Code step 55: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
split_counts = Counter(r["split"] for r in aligned_records)
scene_split_counts = Counter(
    (r["scene"], r["split"]) for r in aligned_records
)

print("\nOfficial split counts")

### Code step 56: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for split in ["train", "val", "test"]:
    print(f"  {split}: {split_counts.get(split, 0):,}")

### Code step 57: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nCounts by scene and split")

### Code step 58: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for scene in sorted(SELECTED_SCENES):
    values = [
        scene_split_counts[(scene, split)]
        for split in ["train", "val", "test"]
    ]
    print(
        f"  {scene:20s} "
        f"train={values[0]:4d} "
        f"val={values[1]:4d} "
        f"test={values[2]:4d}"
    )

### Code step 59: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
filenames_aligned = [r["filename"] for r in aligned_records]

### Code step 60: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(filenames_aligned) == len(set(filenames_aligned))

### Code step 61: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert all(len(r["raster_ids"]) == NUM_PATCHES for r in aligned_records)

### Code step 62: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert all(all(0 <= idx < len(vocab_by_id) for idx in r["raster_ids"])
           for r in aligned_records)

### Code step 63: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
ALIGNED_JSON = OUTPUT_DIR / "module6e_aligned_dataset.json"

### Code step 64: Execute align raster records with rsicd metadata and splits

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
with open(ALIGNED_JSON, "w", encoding="utf-8") as f:
    json.dump(aligned_records, f, indent=2, ensure_ascii=False)

### Code step 65: Align raster records with RSICD metadata and splits

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nSaved aligned dataset to:", ALIGNED_JSON)
print("All alignment assertions passed.")

## 5. Build caption tokenizer from training captions only

Creates a word-level vocabulary using training captions only. PAD=0, UNK=1, BOS=2, EOS=3.

### Code step 66: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

### Code step 67: Define tokenize_caption

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def tokenize_caption(text):
    """Lowercase and tokenize captions into words and punctuation."""
    text = str(text).lower().strip()
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

### Code step 68: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
train_records = [
    record for record in aligned_records
    if record["split"] == "train"
]

val_records = [
    record for record in aligned_records
    if record["split"] == "val"
]

test_records = [
    record for record in aligned_records
    if record["split"] == "test"
]

### Code step 69: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(train_records) == 2244

### Code step 70: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(val_records) == 281

### Code step 71: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(test_records) == 281

### Code step 72: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
token_counts = Counter()

### Code step 73: Execute build caption vocabulary from training captions only

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for record in train_records:
    for caption in record["captions"]:
        token_counts.update(tokenize_caption(caption))

### Code step 74: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
ordered_tokens = sorted(
    token_counts.items(),
    key=lambda item: (-item[1], item[0])
)

itos = [
    PAD_TOKEN,
    UNK_TOKEN,
    BOS_TOKEN,
    EOS_TOKEN,
] + [
    token for token, _ in ordered_tokens
    if token not in {PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN}
]

stoi = {token: idx for idx, token in enumerate(itos)}

### Code step 75: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert stoi[PAD_TOKEN] == PAD_ID

### Code step 76: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert stoi[UNK_TOKEN] == UNK_ID

### Code step 77: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert stoi[BOS_TOKEN] == BOS_ID

### Code step 78: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert stoi[EOS_TOKEN] == EOS_ID

### Code step 79: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
train_caption_lengths = [
    len(tokenize_caption(caption))
    for record in train_records
    for caption in record["captions"]
]

MAX_CAPTION_LEN = 64

print("Caption vocabulary summary")
print("---------------------------")
print("Training images:", len(train_records))
print(
    "Training captions:",
    sum(len(r["captions"]) for r in train_records)
)
print("Vocabulary size:", len(itos))
print("Maximum caption length:", max(train_caption_lengths))
print(
    "Captions longer than the configured limit:",
    sum(length + 2 > MAX_CAPTION_LEN for length in train_caption_lengths)
)
print("Special token IDs:")
print("  PAD:", PAD_ID)
print("  UNK:", UNK_ID)
print("  BOS:", BOS_ID)
print("  EOS:", EOS_ID)

print("\nMost frequent tokens:")

### Code step 80: Execute build caption vocabulary from training captions only

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for token, count in ordered_tokens[:20]:
    print(f"  {token}: {count}")

### Code step 81: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
TOKENIZER_PATH = OUTPUT_DIR / "caption_tokenizer.json"

### Code step 82: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
tokenizer_data = {
    "token_to_id": stoi,
    "id_to_token": itos,
    "special_tokens": {
        "PAD": PAD_ID,
        "UNK": UNK_ID,
        "BOS": BOS_ID,
        "EOS": EOS_ID,
    },
    "max_caption_len": MAX_CAPTION_LEN,
    "tokenization": "lowercase word-level with punctuation",
    "vocabulary_source": "training captions only",
    "training_image_count": len(train_records),
    "training_caption_count": sum(
        len(r["captions"]) for r in train_records
    ),
}

### Code step 83: Execute build caption vocabulary from training captions only

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
with open(TOKENIZER_PATH, "w", encoding="utf-8") as f:
    json.dump(tokenizer_data, f, indent=2, ensure_ascii=False)

### Code step 84: Build caption vocabulary from training captions only

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nSaved tokenizer to:", TOKENIZER_PATH)

## 6. Dataset and DataLoaders

Loads images and raster IDs. Training samples one caption per image; validation/test retain all five references. A custom collate function preserves batch-major reference lists.

### Code step 85: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

### Code step 86: Define encode_caption

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def encode_caption(caption, max_len=MAX_CAPTION_LEN):
    """Encode BOS + word tokens + EOS, then pad to max_len."""
    tokens = tokenize_caption(caption)
    token_ids = [BOS_ID]
    token_ids.extend(stoi.get(token, UNK_ID) for token in tokens)
    token_ids.append(EOS_ID)

    if len(token_ids) > max_len:
        token_ids = token_ids[:max_len]
        token_ids[-1] = EOS_ID

    valid_length = len(token_ids)
    token_ids.extend([PAD_ID] * (max_len - valid_length))

    return torch.tensor(token_ids, dtype=torch.long), valid_length

### Code step 87: Define ERICFusionCaptionDataset

This cell defines one class only. Run it before the cells that instantiate or call it.

In [ ]:
class ERICFusionCaptionDataset(Dataset):
    def __init__(self, records, split, transform=None):
        self.records = list(records)
        self.split = split
        self.transform = transform

        assert split in {"train", "val", "test"}
        assert all(r["split"] == split for r in self.records)

        for record in self.records:
            assert len(record["raster_ids"]) == NUM_PATCHES
            assert len(record["captions"]) == 5, (
                f"Expected five references for {record['filename']}, "
                f"got {len(record['captions'])}"
            )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]

        with Image.open(record["image_path"]) as img:
            image = img.convert("RGB")
            if self.transform is not None:
                image = self.transform(image)

        raster_ids = torch.tensor(
            record["raster_ids"], dtype=torch.long
        )

        if self.split == "train":
            # Sample one reference caption for this training access.
            caption = random.choice(record["captions"])
            caption_ids, caption_length = encode_caption(caption)

            return {
                "filename": record["filename"],
                "scene": record["scene"],
                "image": image,
                "raster_ids": raster_ids,
                "caption_ids": caption_ids,
                "caption_length": caption_length,
                "needs_review": record["needs_review"],
            }

        # Keep all five captions independently for evaluation.
        encoded_references = [
            encode_caption(caption)
            for caption in record["captions"]
        ]

        reference_ids = torch.stack([
            item[0] for item in encoded_references
        ])

        reference_lengths = torch.tensor(
            [item[1] for item in encoded_references],
            dtype=torch.long,
        )

        return {
            "filename": record["filename"],
            "scene": record["scene"],
            "image": image,
            "raster_ids": raster_ids,
            "reference_ids": reference_ids,
            "reference_lengths": reference_lengths,
            "references": list(record["captions"]),
            "needs_review": record["needs_review"],
        }

### Code step 88: Define eric_collate_fn

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def eric_collate_fn(batch):
    """
    Stack tensors while keeping nested reference captions as
    a list of five captions per image.
    """
    result = {}

    tensor_keys = [
        key for key in batch[0]
        if torch.is_tensor(batch[0][key])
    ]

    for key in tensor_keys:
        result[key] = torch.stack([item[key] for item in batch])

    # Keep strings and nested caption lists in batch-major order.
    for key in batch[0]:
        if key not in tensor_keys:
            result[key] = [item[key] for item in batch]

    return result

### Code step 89: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
train_dataset = ERICFusionCaptionDataset(
    train_records, split="train", transform=IMAGE_TRANSFORM
)
val_dataset = ERICFusionCaptionDataset(
    val_records, split="val", transform=IMAGE_TRANSFORM
)
test_dataset = ERICFusionCaptionDataset(
    test_records, split="test", transform=IMAGE_TRANSFORM
)

### Code step 90: Define seed_worker

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

### Code step 91: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator,
    drop_last=False,
    collate_fn=eric_collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    collate_fn=eric_collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    collate_fn=eric_collate_fn,
)


# ----------------------------
# Dataset size checks
# ----------------------------
print("Dataset sizes")
print("-------------")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

### Code step 92: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(train_dataset) == 2244

### Code step 93: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(val_dataset) == 281

### Code step 94: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(test_dataset) == 281

### Code step 95: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
train_batch = next(iter(train_loader))

print("\nTraining batch shapes")
print("----------------------")

### Code step 96: Execute pytorch dataset and dataloaders (corrected)

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for key, value in train_batch.items():
    if torch.is_tensor(value):
        print(f"{key}: {tuple(value.shape)}")
    elif isinstance(value, list):
        print(f"{key}: list of length {len(value)}")

### Code step 97: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert train_batch["image"].shape == (
    BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE
)

### Code step 98: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert train_batch["raster_ids"].shape == (
    BATCH_SIZE, NUM_PATCHES
)

### Code step 99: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert train_batch["caption_ids"].shape == (
    BATCH_SIZE, MAX_CAPTION_LEN
)

### Code step 100: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert train_batch["raster_ids"].min().item() >= 0

### Code step 101: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert train_batch["raster_ids"].max().item() < len(vocab_by_id)

### Code step 102: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
val_batch = next(iter(val_loader))

print("\nValidation batch shapes")
print("-----------------------")
print("Images:", tuple(val_batch["image"].shape))
print("Raster IDs:", tuple(val_batch["raster_ids"].shape))
print("Reference IDs:", tuple(val_batch["reference_ids"].shape))
print("Reference lengths:", tuple(val_batch["reference_lengths"].shape))
print("First validation filename:", val_batch["filename"][0])
print("First validation scene:", val_batch["scene"][0])
print("References for first image:", len(val_batch["references"][0]))

### Code step 103: Execute pytorch dataset and dataloaders (corrected)

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for i, refs in enumerate(val_batch["references"]):
    assert len(refs) == 5, (
        f"Batch item {i} has {len(refs)} references instead of 5."
    )

### Code step 104: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert val_batch["reference_ids"].shape == (
    len(val_batch["filename"]), 5, MAX_CAPTION_LEN
)

### Code step 105: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert val_batch["reference_lengths"].shape == (
    len(val_batch["filename"]), 5
)

### Code step 106: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(val_batch["references"]) == len(val_batch["filename"])

### Code step 107: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(val_batch["references"][0]) == 5

### Code step 108: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nFirst image's five references:")

### Code step 109: Execute pytorch dataset and dataloaders (corrected)

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for i, caption in enumerate(val_batch["references"][0], start=1):
    print(f"  {i}. {caption}")

### Code step 110: PyTorch dataset and DataLoaders (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nDataset and DataLoader checks passed.")

## 7. Image encoder and spatial fusion

ViT-Base/32 outputs 49 image patch tokens. Raster embeddings receive 2D position information and are fused with the corresponding image token.

### Code step 111: ViT-Base/32 encoder and spatial fusion architecture

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import timm
import torch
import torch.nn as nn

### Code step 112: Define SpatialImageRasterFusion

This cell defines one class only. Run it before the cells that instantiate or call it.

In [ ]:
class SpatialImageRasterFusion(nn.Module):
    """
    Fuses each ViT spatial patch with the raster label at the
    corresponding 7x7 grid position.

    Input:
        image_tokens: [B, 49, 768]
        raster_ids:   [B, 49]

    Output:
        fused_tokens: [B, 49, 512]
    """

    def __init__(
        self,
        image_dim=768,
        raster_vocab_size=210,
        raster_dim=512,
        fusion_dim=512,
        grid_size=7,
        dropout=0.1,
    ):
        super().__init__()

        self.grid_size = grid_size
        self.num_patches = grid_size * grid_size

        self.raster_embedding = nn.Embedding(
            raster_vocab_size, raster_dim
        )

        # Explicit 2D positional information for the raster cells.
        self.row_embedding = nn.Embedding(grid_size, raster_dim)
        self.col_embedding = nn.Embedding(grid_size, raster_dim)

        row_ids = torch.arange(grid_size).repeat_interleave(grid_size)
        col_ids = torch.arange(grid_size).repeat(grid_size)

        self.register_buffer("row_ids", row_ids, persistent=False)
        self.register_buffer("col_ids", col_ids, persistent=False)

        # Align and fuse each image patch with its corresponding
        # semantic raster cell.
        self.fusion_projection = nn.Sequential(
            nn.Linear(image_dim + raster_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, image_tokens, raster_ids):
        assert image_tokens.ndim == 3
        assert raster_ids.ndim == 2
        assert image_tokens.shape[1] == self.num_patches
        assert raster_ids.shape[1] == self.num_patches

        raster_features = self.raster_embedding(raster_ids)

        row_features = self.row_embedding(self.row_ids)
        col_features = self.col_embedding(self.col_ids)
        position_features = row_features + col_features

        raster_features = raster_features + position_features.unsqueeze(0)

        combined = torch.cat(
            [image_tokens, raster_features], dim=-1
        )

        fused_tokens = self.fusion_projection(combined)

        return fused_tokens

### Code step 113: Define ERICModule6eEncoder

This cell defines one class only. Run it before the cells that instantiate or call it.

In [ ]:
class ERICModule6eEncoder(nn.Module):
    """
    Image encoder and raster-fusion memory builder.

    Output memory:
        [B, 50, 512]
        Token 0: projected global CLS feature
        Tokens 1-49: spatially fused image-raster features
    """

    def __init__(
        self,
        raster_vocab_size,
        fusion_dim=512,
        dropout=0.1,
        pretrained=True,
        unfreeze_last_n_blocks=2,
    ):
        super().__init__()

        self.image_encoder = timm.create_model(
            "vit_base_patch32_224",
            pretrained=pretrained,
            num_classes=0,
        )

        # This model must produce 7x7 spatial patches at 224x224.
        assert self.image_encoder.patch_embed.patch_size == (32, 32), (
            "Expected ViT-Base/32 with 32x32 patches."
        )

        self.image_dim = self.image_encoder.num_features
        assert self.image_dim == 768, (
            f"Expected image dimension 768, got {self.image_dim}."
        )

        # Freeze all image encoder parameters initially.
        for parameter in self.image_encoder.parameters():
            parameter.requires_grad = False

        # Fine-tune the final N Transformer blocks.
        blocks = self.image_encoder.blocks
        assert 0 <= unfreeze_last_n_blocks <= len(blocks)

        for block in blocks[-unfreeze_last_n_blocks:]:
            for parameter in block.parameters():
                parameter.requires_grad = True

        # Also fine-tune the final normalization layer.
        for parameter in self.image_encoder.norm.parameters():
            parameter.requires_grad = True

        self.global_projection = nn.Sequential(
            nn.Linear(self.image_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
        )

        self.fusion = SpatialImageRasterFusion(
            image_dim=self.image_dim,
            raster_vocab_size=raster_vocab_size,
            raster_dim=fusion_dim,
            fusion_dim=fusion_dim,
            grid_size=GRID_SIZE,
            dropout=dropout,
        )

    def forward(self, images, raster_ids):
        # timm forward_features returns CLS + spatial patch tokens.
        features = self.image_encoder.forward_features(images)

        assert features.ndim == 3
        assert features.shape[1] == NUM_PATCHES + 1, (
            f"Expected 50 ViT tokens (CLS + 49 patches), "
            f"got {features.shape[1]}."
        )

        cls_token = features[:, 0, :]
        image_patch_tokens = features[:, 1:, :]

        global_token = self.global_projection(cls_token).unsqueeze(1)

        fused_spatial_tokens = self.fusion(
            image_patch_tokens, raster_ids
        )

        memory = torch.cat(
            [global_token, fused_spatial_tokens], dim=1
        )

        assert memory.shape[1:] == (NUM_PATCHES + 1, D_MODEL)

        return memory

### Code step 114: ViT-Base/32 encoder and spatial fusion architecture

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
encoder = ERICModule6eEncoder(
    raster_vocab_size=len(vocab_by_id),
    fusion_dim=D_MODEL,
    dropout=DROPOUT,
    pretrained=True,
    unfreeze_last_n_blocks=2,
).to(device)

### Code step 115: Define count_parameters

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def count_parameters(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(
        p.numel() for p in module.parameters()
        if p.requires_grad
    )
    return total, trainable

### Code step 116: ViT-Base/32 encoder and spatial fusion architecture

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
total_params, trainable_params = count_parameters(encoder)

print("Module 6e encoder initialized")
print("-----------------------------")
print("Encoder:", "ViT-Base/32")
print("Image feature dimension:", encoder.image_dim)
print("Grid:", f"{GRID_SIZE} x {GRID_SIZE}")
print("Spatial patches:", NUM_PATCHES)
print("Raster vocabulary size:", len(vocab_by_id))
print("Fused spatial dimension:", D_MODEL)
print("Memory tokens:", NUM_PATCHES + 1)
print("Total encoder parameters:", f"{total_params:,}")
print("Trainable encoder parameters:", f"{trainable_params:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.2f}%"
)

trainable_blocks = [
    i for i, block in enumerate(encoder.image_encoder.blocks)
    if any(p.requires_grad for p in block.parameters())
]
print("Fine-tuned ViT block indices:", trainable_blocks)

### Code step 117: ViT-Base/32 encoder and spatial fusion architecture

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert trainable_blocks == [10, 11]

### Code step 118: ViT-Base/32 encoder and spatial fusion architecture

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert encoder.image_encoder.norm.weight.requires_grad

### Code step 119: ViT-Base/32 encoder and spatial fusion architecture

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert encoder.fusion.raster_embedding.weight.requires_grad

### Code step 120: ViT-Base/32 encoder and spatial fusion architecture

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nEncoder architecture checks passed.")

## 8. Transformer caption decoder

An autoregressive Transformer decoder predicts the next caption token using causal self-attention and cross-attention to image/raster memory.

### Code step 121: Transformer caption decoder

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import torch
import torch.nn as nn
import math

### Code step 122: Define CaptionTransformerDecoder

This cell defines one class only. Run it before the cells that instantiate or call it.

In [ ]:
class CaptionTransformerDecoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=512,
        num_layers=3,
        num_heads=8,
        dim_feedforward=2048,
        dropout=0.1,
        max_len=64,
    ):
        super().__init__()

        self.d_model = d_model
        self.max_len = max_len

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_ID,
        )

        self.position_embedding = nn.Embedding(
            max_len,
            d_model,
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )

        self.output_projection = nn.Linear(
            d_model, vocab_size
        )

        self.dropout = nn.Dropout(dropout)

        # Positions are [0, 1, ..., max_len-1].
        self.register_buffer(
            "position_ids",
            torch.arange(max_len).unsqueeze(0),
            persistent=False,
        )

    def forward(self, caption_input, memory):
        """
        caption_input: [B, T]
        memory:        [B, 50, 512]
        returns logits [B, T, vocab_size]
        """
        batch_size, seq_len = caption_input.shape

        assert seq_len <= self.max_len
        assert memory.shape[0] == batch_size
        assert memory.shape[-1] == self.d_model

        positions = self.position_ids[:, :seq_len]

        target = (
            self.token_embedding(caption_input) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.dropout(target)

        # True above the diagonal blocks access to future tokens.
        causal_mask = torch.triu(
            torch.ones(
                seq_len, seq_len,
                device=caption_input.device,
                dtype=torch.bool,
            ),
            diagonal=1,
        )

        # Ignore padded target positions during self-attention.
        target_padding_mask = caption_input.eq(PAD_ID)

        decoded = self.decoder(
            tgt=target,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=target_padding_mask,
        )

        logits = self.output_projection(decoded)

        return logits

### Code step 123: Transformer caption decoder

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
decoder = CaptionTransformerDecoder(
    vocab_size=len(itos),
    d_model=D_MODEL,
    num_layers=NUM_DECODER_LAYERS,
    num_heads=NUM_HEADS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_len=MAX_CAPTION_LEN,
).to(device)


# ----------------------------
# Parameter and configuration checks
# ----------------------------
decoder_total, decoder_trainable = count_parameters(decoder)

print("Caption decoder initialized")
print("----------------------------")
print("Vocabulary size:", len(itos))
print("Decoder layers:", NUM_DECODER_LAYERS)
print("Hidden dimension:", D_MODEL)
print("Attention heads:", NUM_HEADS)
print("Feed-forward dimension:", DIM_FEEDFORWARD)
print("Maximum caption length:", MAX_CAPTION_LEN)
print("Total decoder parameters:", f"{decoder_total:,}")
print("Trainable decoder parameters:", f"{decoder_trainable:,}")


# ----------------------------
# Forward-pass smoke test
# ----------------------------
encoder.eval()
decoder.eval()

### Code step 124: Execute transformer caption decoder

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
with torch.no_grad():
    sample_images = train_batch["image"][:2].to(device)
    sample_rasters = train_batch["raster_ids"][:2].to(device)
    sample_captions = train_batch["caption_ids"][:2].to(device)

    sample_memory = encoder(sample_images, sample_rasters)

    # Teacher forcing: input excludes the final target token.
    sample_input = sample_captions[:, :-1]
    sample_logits = decoder(sample_input, sample_memory)

### Code step 125: Transformer caption decoder

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nSmoke-test shapes")
print("-----------------")
print("Images:", tuple(sample_images.shape))
print("Raster IDs:", tuple(sample_rasters.shape))
print("Encoder memory:", tuple(sample_memory.shape))
print("Decoder input:", tuple(sample_input.shape))
print("Decoder logits:", tuple(sample_logits.shape))

### Code step 126: Transformer caption decoder

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert sample_memory.shape == (2, NUM_PATCHES + 1, D_MODEL)

### Code step 127: Transformer caption decoder

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert sample_logits.shape == (
    2, MAX_CAPTION_LEN - 1, len(itos)
)

### Code step 128: Transformer caption decoder

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert torch.isfinite(sample_logits).all()

### Code step 129: Transformer caption decoder

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nDecoder architecture and forward-pass checks passed.")

## 9. Complete model and training loss

Connects the encoder and decoder. Cross-entropy ignores padding tokens. Smoke tests check shapes before training.

### Code step 130: Complete fusion captioning model and training loss

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import torch
import torch.nn as nn

### Code step 131: Define ERICModule6e

This cell defines one class only. Run it before the cells that instantiate or call it.

In [ ]:
class ERICModule6e(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, raster_ids, caption_input):
        """
        images:        [B, 3, 224, 224]
        raster_ids:    [B, 49]
        caption_input: [B, T]

        Returns:
            logits: [B, T, vocabulary_size]
        """
        memory = self.encoder(images, raster_ids)
        logits = self.decoder(caption_input, memory)
        return logits

### Code step 132: Complete fusion captioning model and training loss

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
model = ERICModule6e(
    encoder=encoder,
    decoder=decoder,
).to(device)

# Ignore PAD positions when computing cross-entropy.
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
)

# ----------------------------
# Verify trainable parameters
# ----------------------------
total_params, trainable_params = count_parameters(model)

print("Complete Module 6e model")
print("------------------------")
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.2f}%"
)
print("Loss:", criterion)


# ----------------------------
# Teacher-forcing and loss smoke test
# ----------------------------
model.train()

sample_images = train_batch["image"][:2].to(device)
sample_rasters = train_batch["raster_ids"][:2].to(device)
sample_caption_ids = train_batch["caption_ids"][:2].to(device)

# For BOS, w1, w2, EOS, PAD...
# decoder input:  BOS, w1, w2, ...
# target:         w1,  w2, EOS, ...
caption_input = sample_caption_ids[:, :-1]
caption_target = sample_caption_ids[:, 1:]

optimizer_test = None

logits = model(
    sample_images,
    sample_rasters,
    caption_input,
)

loss = criterion(
    logits.reshape(-1, logits.shape[-1]),
    caption_target.reshape(-1),
)

print("\nTraining smoke test")

### Code step 133: Complete fusion captioning model and training loss

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("-------------------")
print("Caption input:", tuple(caption_input.shape))
print("Caption target:", tuple(caption_target.shape))
print("Logits:", tuple(logits.shape))
print("Loss:", float(loss.detach().cpu()))

### Code step 134: Complete fusion captioning model and training loss

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert logits.shape[:2] == caption_target.shape

### Code step 135: Complete fusion captioning model and training loss

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert torch.isfinite(loss)

### Code step 136: Complete fusion captioning model and training loss

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert loss.item() > 0

### Code step 137: Complete fusion captioning model and training loss

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nComplete model and loss checks passed.")

## 10. Optimizer, mixed precision, scheduler, and checkpoints

Configures AdamW, mixed precision, gradient clipping, ReduceLROnPlateau, and checkpoint paths.

### Code step 138: Optimizer, AMP, scheduler, and checkpoint settings

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import copy
import json
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ----------------------------
# Training configuration
# ----------------------------
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
EARLY_STOPPING_PATIENCE = 5
LR_SCHEDULER_PATIENCE = 2
LR_SCHEDULER_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-6

# Enable mixed precision on CUDA.
USE_AMP = device.type == "cuda"
AMP_DTYPE = torch.float16

# ----------------------------
# Optimizer
# ----------------------------
trainable_parameters = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer = AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=LR_SCHEDULER_FACTOR,
    patience=LR_SCHEDULER_PATIENCE,
    min_lr=MIN_LEARNING_RATE,
)

# GradScaler is needed for stable FP16 training.
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

# ----------------------------
# Checkpoint paths and metadata
# ----------------------------
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_model.pt"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_model.pt"
TRAINING_HISTORY_PATH = OUTPUT_DIR / "training_history.json"

### Code step 139: Optimizer, AMP, scheduler, and checkpoint settings

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
CONFIG_PATH = OUTPUT_DIR / "model_config.json"

### Code step 140: Optimizer, AMP, scheduler, and checkpoint settings

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
model_config = {
    "module": "ERIC_Module6e",
    "architecture": "ViT-Base/32 + spatial image-raster fusion + Transformer decoder",
    "image_size": IMAGE_SIZE,
    "patch_size": PATCH_SIZE,
    "grid_size": GRID_SIZE,
    "num_patches": NUM_PATCHES,
    "raster_vocab_size": len(vocab_by_id),
    "caption_vocab_size": len(itos),
    "d_model": D_MODEL,
    "decoder_layers": NUM_DECODER_LAYERS,
    "attention_heads": NUM_HEADS,
    "dim_feedforward": DIM_FEEDFORWARD,
    "dropout": DROPOUT,
    "max_caption_len": MAX_CAPTION_LEN,
    "pad_id": PAD_ID,
    "unk_id": UNK_ID,
    "bos_id": BOS_ID,
    "eos_id": EOS_ID,
    "fine_tuned_vit_blocks": [10, 11],
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "max_epochs": NUM_EPOCHS,
    "mixed_precision": USE_AMP,
    "seed": SEED,
    "selected_scenes": sorted(SELECTED_SCENES),
    "official_split_counts": {
        "train": len(train_dataset),
        "val": len(val_dataset),
        "test": len(test_dataset),
    },
}

### Code step 141: Execute optimizer, amp, scheduler, and checkpoint settings

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=2)

### Code step 142: Optimizer, AMP, scheduler, and checkpoint settings

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
training_history = {
    "train_loss": [],
    "val_loss": [],
    "learning_rate": [],
    "epoch_seconds": [],
}

print("Training configuration ready")
print("-----------------------------")
print("Optimizer: AdamW")
print("Initial learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Gradient clipping norm:", GRAD_CLIP_NORM)
print("Mixed precision enabled:", USE_AMP)
print("AMP dtype:", str(AMP_DTYPE) if USE_AMP else "disabled")
print("Early-stopping patience:", EARLY_STOPPING_PATIENCE)
print("LR scheduler: ReduceLROnPlateau")
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Last checkpoint:", LAST_CHECKPOINT_PATH)
print("Model configuration:", CONFIG_PATH)

### Code step 143: Optimizer, AMP, scheduler, and checkpoint settings

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(trainable_parameters) > 0

### Code step 144: Optimizer, AMP, scheduler, and checkpoint settings

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert model_config["official_split_counts"] == {
    "train": 2244,
    "val": 281,
    "test": 281,
}

### Code step 145: Optimizer, AMP, scheduler, and checkpoint settings

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nTraining configuration checks passed.")

## 11. Training and validation loop

Uses teacher forcing for training and a deterministic reference for validation loss. Early stopping is based on validation loss.

### Code step 146: Training and validation loop

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import time
import json
import torch
from tqdm.auto import tqdm

### Code step 147: Define run_epoch

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    """
    Run one training or validation epoch.

    Training uses teacher forcing:
      input  = caption_ids[:, :-1]
      target = caption_ids[:, 1:]

    Padding tokens are ignored by criterion(ignore_index=PAD_ID).
    """
    is_training = optimizer is not None
    model.train(is_training)

    total_loss_sum = 0.0
    total_valid_tokens = 0
    total_images = 0

    progress = tqdm(
        loader,
        desc="Train" if is_training else "Validation",
        leave=False,
    )

    for batch in progress:
        images = batch["image"].to(device, non_blocking=True)
        raster_ids = batch["raster_ids"].to(
            device, non_blocking=True
        )

        if is_training:
            caption_ids = batch["caption_ids"].to(
                device, non_blocking=True
            )
        else:
            # Use one reference per image for validation loss.
            # Select reference 0 deterministically.
            caption_ids = batch["reference_ids"][:, 0, :].to(
                device, non_blocking=True
            )

        caption_input = caption_ids[:, :-1]
        caption_target = caption_ids[:, 1:]

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            with torch.autocast(
                device_type=device.type,
                dtype=AMP_DTYPE,
                enabled=USE_AMP,
            ):
                logits = model(
                    images,
                    raster_ids,
                    caption_input,
                )

                # Sum token losses, then normalize by the number
                # of non-padding target tokens.
                loss_sum = torch.nn.functional.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    caption_target.reshape(-1),
                    ignore_index=PAD_ID,
                    reduction="sum",
                )

                valid_tokens = caption_target.ne(PAD_ID).sum()
                loss = loss_sum / valid_tokens.clamp_min(1)

            if is_training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    [
                        p for p in model.parameters()
                        if p.requires_grad and p.grad is not None
                    ],
                    GRAD_CLIP_NORM,
                )

                scaler.step(optimizer)
                scaler.update()

        batch_tokens = int(valid_tokens.item())
        total_loss_sum += float(loss_sum.detach().item())
        total_valid_tokens += batch_tokens
        total_images += images.size(0)

        current_loss = total_loss_sum / max(total_valid_tokens, 1)
        progress.set_postfix(
            loss=f"{current_loss:.4f}",
            tokens=total_valid_tokens,
        )

    epoch_loss = total_loss_sum / max(total_valid_tokens, 1)

    return {
        "loss": epoch_loss,
        "valid_tokens": total_valid_tokens,
        "images": total_images,
    }

### Code step 148: Training and validation loop

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

training_history = {
    "train_loss": [],
    "val_loss": [],
    "learning_rate": [],
    "epoch_seconds": [],
}

print("Starting Module 6e training")
print("----------------------------")
print("Maximum epochs:", NUM_EPOCHS)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Device:", device)
print("Mixed precision:", USE_AMP)

### Code step 149: Execute training and validation loop

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    train_result = run_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
    )

    val_result = run_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
        optimizer=None,
        scaler=None,
    )

    train_loss = train_result["loss"]
    val_loss = val_result["loss"]
    epoch_seconds = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]["lr"]

    scheduler.step(val_loss)

    training_history["train_loss"].append(train_loss)
    training_history["val_loss"].append(val_loss)
    training_history["learning_rate"].append(current_lr)
    training_history["epoch_seconds"].append(epoch_seconds)

    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_val_loss": best_val_loss,
                "model_config": model_config,
                "token_to_id": stoi,
                "id_to_token": itos,
            },
            BEST_CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    # Save the most recent training state for recovery.
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "epochs_without_improvement": epochs_without_improvement,
            "model_config": model_config,
            "training_history": training_history,
        },
        LAST_CHECKPOINT_PATH,
    )

    with open(TRAINING_HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(training_history, f, indent=2)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {val_loss:.4f} | "
        f"best val: {best_val_loss:.4f} (epoch {best_epoch}) | "
        f"lr: {current_lr:.2e} | "
        f"time: {epoch_seconds:.1f}s"
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"\nEarly stopping: validation loss did not improve for "
            f"{EARLY_STOPPING_PATIENCE} consecutive epochs."
        )
        break

### Code step 150: Training and validation loop

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nTraining finished")
print("------------------")
print("Best epoch:", best_epoch)
print("Best validation loss:", f"{best_val_loss:.4f}")
print("Best checkpoint exists:", BEST_CHECKPOINT_PATH.exists())
print("Last checkpoint exists:", LAST_CHECKPOINT_PATH.exists())
print("Training history:", TRAINING_HISTORY_PATH)

### Code step 151: Training and validation loop

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert BEST_CHECKPOINT_PATH.exists()

### Code step 152: Training and validation loop

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert best_epoch >= 1

## 12. Restore the best validation checkpoint

Loads the model checkpoint with the best validation loss for generation and evaluation.

### Code step 153: Restore the best validation checkpoint

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import torch

### Code step 154: Restore the best validation checkpoint

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert BEST_CHECKPOINT_PATH.exists(), (
    f"Best checkpoint not found: {BEST_CHECKPOINT_PATH}"
)

### Code step 155: Restore the best validation checkpoint

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

best_epoch_loaded = checkpoint["epoch"]
best_val_loss_loaded = checkpoint["best_val_loss"]

print("Best checkpoint loaded")
print("-----------------------")
print("Checkpoint:", BEST_CHECKPOINT_PATH)
print("Best epoch:", best_epoch_loaded)
print("Best validation loss:", f"{best_val_loss_loaded:.4f}")
print("Model device:", next(model.parameters()).device)

### Code step 156: Restore the best validation checkpoint

This cell contains a small setup, validation, or execution step for the current section.

### Code step 157: Restore the best validation checkpoint

This cell contains a small setup, validation, or execution step for the current section.

### Code step 158: Restore the best validation checkpoint

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nCheckpoint restoration verified.")

## 13. Greedy caption generation

Generates a caption token-by-token from image and raster inputs until EOS or maximum length.

### Code step 159: Greedy caption generation

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import torch

### Code step 160: Define generate_caption

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def generate_caption(
    model,
    image,
    raster_ids,
    max_len=MAX_CAPTION_LEN,
):
    """
    Generate one caption using the image and its semantic raster.

    Args:
        image: [3, 224, 224] tensor or [1, 3, 224, 224] tensor
        raster_ids: [49] tensor or [1, 49] tensor

    Returns:
        Generated caption string.
    """
    model.eval()

    if image.ndim == 3:
        image = image.unsqueeze(0)

    if raster_ids.ndim == 1:
        raster_ids = raster_ids.unsqueeze(0)

    image = image.to(device)
    raster_ids = raster_ids.to(device)

    assert image.shape[0] == 1
    assert raster_ids.shape == (1, NUM_PATCHES)

    # Encode image and raster once.
    memory = model.encoder(image, raster_ids)

    # Start autoregressive generation with BOS.
    generated_ids = [BOS_ID]

    for _ in range(max_len - 1):
        decoder_input = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=device,
        )

        # Inference without mixed-precision dependence for decoding.
        with torch.autocast(
            device_type=device.type,
            dtype=AMP_DTYPE,
            enabled=USE_AMP,
        ):
            logits = model.decoder(decoder_input, memory)

        # Select the next token from the final decoder position.
        next_token_id = int(
            logits[0, -1].argmax(dim=-1).item()
        )

        if next_token_id == EOS_ID:
            break

        # Avoid emitting special tokens as caption text.
        if next_token_id in {PAD_ID, BOS_ID}:
            break

        generated_ids.append(next_token_id)

    # Convert generated word IDs to tokens.
    words = [
        itos[token_id]
        for token_id in generated_ids[1:]
        if token_id not in {PAD_ID, BOS_ID, EOS_ID}
    ]

    # Join punctuation without inserting spaces before it.
    caption = ""
    no_space_before = {".", ",", "!", "?", ";", ":", ")"}
    no_space_after = {"("}

    for word in words:
        if not caption:
            caption = word
        elif word in no_space_before:
            caption += word
        elif caption.split()[-1] in no_space_after:
            caption += word
        else:
            caption += " " + word

    return caption.strip()

### Code step 161: Greedy caption generation

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
sample = val_dataset[0]

predicted_caption = generate_caption(
    model,
    sample["image"],
    sample["raster_ids"],
)

print("Caption generation smoke test")
print("-----------------------------")
print("Filename:", sample["filename"])
print("Scene:", sample["scene"])
print("Generated caption:", predicted_caption)
print("Reference captions:")

### Code step 162: Execute greedy caption generation

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for i, reference in enumerate(sample["references"], start=1):
    print(f"  {i}. {reference}")

### Code step 163: Greedy caption generation

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert isinstance(predicted_caption, str)

### Code step 164: Greedy caption generation

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(predicted_caption) > 0

### Code step 165: Greedy caption generation

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert len(sample["references"]) == 5

### Code step 166: Greedy caption generation

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("\nCaption generation check passed.")

## 14. Qualitative inspection

Shows original images, generated captions, and all five references to inspect generic wording, omissions, and hallucinations.

### Code step 167: Qualitative visualization (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

# Number of validation images to display
N = 8
SAMPLE_SEED = 42

### Code step 168: Qualitative visualization (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert 1 <= N <= len(val_dataset)

### Code step 169: Qualitative visualization (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
record_by_filename = {
    record["filename"]: record
    for record in aligned_records
}

rng = random.Random(SAMPLE_SEED)
sample_indices = rng.sample(range(len(val_dataset)), N)

model.eval()

### Code step 170: Execute qualitative visualization (corrected)

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for sample_number, sample_idx in enumerate(sample_indices, start=1):
    sample = val_dataset[sample_idx]
    filename = sample["filename"]

    # Look up the original image path using its filename.
    metadata_record = record_by_filename[filename]
    image_path = metadata_record["image_path"]

    # Generate using only the image and its raster.
    prediction = generate_caption(
        model,
        sample["image"],
        sample["raster_ids"],
    )

    # Load the original RGB image.
    with Image.open(image_path) as img:
        original_image = img.convert("RGB").copy()

    references = sample["references"]
    assert len(references) == 5

    fig, (ax_image, ax_text) = plt.subplots(
        1, 2,
        figsize=(16, 6),
        gridspec_kw={"width_ratios": [1, 1.45]},
    )

    ax_image.imshow(original_image)
    ax_image.set_title(
        f"{filename}\nScene: {sample['scene']}",
        fontsize=12,
    )
    ax_image.axis("off")

    ax_text.axis("off")

    text_lines = [
        "GENERATED CAPTION",
        prediction,
        "",
        "GROUND-TRUTH REFERENCES",
    ]

    for ref_idx, reference in enumerate(references, start=1):
        text_lines.append(f"{ref_idx}. {reference}")

    ax_text.text(
        0.02, 0.98,
        "\n".join(text_lines),
        transform=ax_text.transAxes,
        va="top",
        ha="left",
        fontsize=10.5,
        linespacing=1.5,
        wrap=True,
    )

    fig.suptitle(
        f"Module 6e — Sample {sample_number}/{N}",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()
    plt.close(fig)

### Code step 171: Qualitative visualization (corrected)

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print(f"Displayed {N} validation samples successfully.")

## 15. Caption evaluation protocol and metric theory

Evaluate validation and test separately using one generated caption and five references per image.

- **BLEU-1 to BLEU-4:** corpus-level n-gram precision with a brevity penalty.
- **METEOR:** aligns candidate and reference words with linguistic matching.
- **ROUGE-L:** measures longest-common-subsequence overlap; this notebook averages the best reference F-score per image.
- **CIDEr:** measures consensus using n-gram informativeness across the reference set.

These metrics quantify text overlap, not spatial correctness directly. Keep tokenization, smoothing, reference handling, and library versions consistent across experimental modules.

### 15.1 Install evaluation dependencies

Run once per fresh Kaggle runtime if these packages are not installed. If installation fails, resolve the package error before continuing.

### Code step 172: Install or configure dependencies

Run this setup command before importing or using the package.

In [ ]:
%pip -q install nltk rouge-score pycocoevalcap

### 15.2 Import evaluation dependencies and define output directory

This cell verifies the metric libraries import successfully and creates a dedicated directory for predictions and scores.

### Code step 173: 15.2 Import evaluation dependencies and define output directory

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
import json
import csv
import time
import re
from collections import OrderedDict
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

SCORE_DIR = OUTPUT_DIR / "evaluation"
SCORE_DIR.mkdir(parents=True, exist_ok=True)
print("Evaluation output directory:", SCORE_DIR)
print("Metric dependencies imported successfully.")

### 15.3 Tokenization and metric computation

BLEU is calculated at corpus level; METEOR and best-reference ROUGE-L are averaged per image; CIDEr is calculated over the reference corpus. Five references per image are required.

### Code step 174: Define metric_tokens

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def metric_tokens(text):
    """Lowercase word and punctuation tokenization."""
    return re.findall(r"\w+|[^\w\s]", str(text).lower(), flags=re.UNICODE)

### Code step 175: Define compute_caption_metrics

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def compute_caption_metrics(predictions, references_by_image):
    """Compute BLEU-1..4, mean METEOR, mean best-reference ROUGE-L, and CIDEr."""
    assert len(predictions) == len(references_by_image) > 0
    assert all(len(refs) == 5 for refs in references_by_image), (
        "Every image must have exactly five reference captions."
    )

    tokenized_predictions = [metric_tokens(p) for p in predictions]
    tokenized_references = [
        [metric_tokens(ref) for ref in refs]
        for refs in references_by_image
    ]

    smoother = SmoothingFunction().method1
    bleu_weights = {
        "BLEU-1": (1.0, 0.0, 0.0, 0.0),
        "BLEU-2": (0.5, 0.5, 0.0, 0.0),
        "BLEU-3": (1/3, 1/3, 1/3, 0.0),
        "BLEU-4": (0.25, 0.25, 0.25, 0.25),
    }
    scores = {}
    for name, weights in bleu_weights.items():
        scores[name] = float(corpus_bleu(
            tokenized_references, tokenized_predictions,
            weights=weights, smoothing_function=smoother
        ))

    meteor_values = [
        meteor_score(refs, pred)
        for pred, refs in zip(tokenized_predictions, tokenized_references)
    ]

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_values = []
    for prediction, references in zip(predictions, references_by_image):
        rouge_values.append(max(
            rouge.score(reference, prediction)["rougeL"].fmeasure
            for reference in references
        ))

    cider_gts, cider_res = OrderedDict(), OrderedDict()
    for idx, (prediction, references) in enumerate(zip(predictions, references_by_image)):
        cider_gts[idx] = references
        cider_res[idx] = [prediction]
    cider_score, _ = Cider().compute_score(cider_gts, cider_res)

    scores["METEOR"] = float(np.mean(meteor_values))
    scores["ROUGE-L"] = float(np.mean(rouge_values))
    scores["CIDEr"] = float(cider_score)
    return scores

### 15.4 Generate captions and save per-image predictions

The JSON schema is compatible with Module 6b: `filename`, `scene`, `prediction`, and `references`. This makes predictions easy to inspect and compare later.

### Code step 176: Define evaluate_split

This cell defines a reusable function. It does not execute the full workflow by itself.

In [ ]:
def evaluate_split(dataset, split_name):
    model.eval()
    prediction_rows = []

    for idx in tqdm(range(len(dataset)), desc=f"Generating {split_name} captions"):
        sample = dataset[idx]
        prediction = generate_caption(model, sample["image"], sample["raster_ids"])
        references = list(sample["references"])
        assert len(references) == 5, (
            f'{sample["filename"]} has {len(references)} references.'
        )
        prediction_rows.append({
            "filename": sample["filename"],
            "scene": sample["scene"],
            "prediction": prediction,
            "references": references,
        })

    metrics = compute_caption_metrics(
        [row["prediction"] for row in prediction_rows],
        [row["references"] for row in prediction_rows],
    )

    prediction_path = SCORE_DIR / f"{split_name}_prediction.json"
    with open(prediction_path, "w", encoding="utf-8") as f:
        json.dump(prediction_rows, f, indent=2, ensure_ascii=False)

    metrics_path = SCORE_DIR / f"{split_name}_metrics.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print(f"\n{split_name.upper()} metrics")
    for name, value in metrics.items():
        print(f"{name:10s}: {value:.6f}")
    print("Predictions saved:", prediction_path)
    print("Metrics saved:    ", metrics_path)
    return metrics, prediction_rows

### 15.5 Evaluate validation and test sets

Use validation results for model-selection decisions. Treat test results as the held-out final estimate and do not tune model settings based on test performance.

### Code step 177: 15.5 Evaluate validation and test sets

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
evaluation_start_time = time.time()

val_metrics_6e, val_predictions_6e = evaluate_split(val_dataset, "validation")
test_metrics_6e, test_predictions_6e = evaluate_split(test_dataset, "test")

print(f"Evaluation duration: {(time.time() - evaluation_start_time) / 60:.2f} minutes")

### 15.6 Save combined metric summaries

Saves both JSON and CSV versions for reporting and comparison with other ERIC modules. Metric implementation consistency must be verified before comparing the scores across modules.

### Code step 178: 15.6 Save combined metric summaries

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
metric_names = [
    "BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4",
    "METEOR", "ROUGE-L", "CIDEr",
]
combined_metrics_6e = {
    "validation": val_metrics_6e,
    "test": test_metrics_6e,
}

combined_json_path = SCORE_DIR / "module6e_metrics.json"

### Code step 179: Execute 15.6 save combined metric summaries

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
with open(combined_json_path, "w", encoding="utf-8") as f:
    json.dump(combined_metrics_6e, f, indent=2)

### Code step 180: 15.6 Save combined metric summaries

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
combined_csv_path = SCORE_DIR / "module6e_metrics.csv"

### Code step 181: Execute 15.6 save combined metric summaries

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
with open(combined_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Split"] + metric_names)
    for split_name, metrics in combined_metrics_6e.items():
        writer.writerow([split_name] + [metrics[name] for name in metric_names])

### Code step 182: 15.6 Save combined metric summaries

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("Images evaluated:", {
    "validation": len(val_predictions_6e),
    "test": len(test_predictions_6e),
})
print("Combined JSON:", combined_json_path)
print("Combined CSV: ", combined_csv_path)

### Code step 183: Execute 15.6 save combined metric summaries

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
for split_name, metrics in combined_metrics_6e.items():
    print(f"\n{split_name.upper()}")
    for name in metric_names:
        print(f"{name:10s}: {metrics[name]:.6f}")

## 16. Interpretation and reporting checklist

After running the notebook, record:
1. The checkpoint epoch selected by validation loss.
2. Validation and test metrics.
3. Representative qualitative examples.
4. Limitations, including unverified pseudo-labels and the validation-to-test gap.
5. A controlled comparison with Modules 6a and 6b using a standardized evaluation pipeline.

Do not claim fusion improves captioning unless the comparison is methodologically fair. For edge deployment, separately report raster-generation latency, caption inference latency, memory use, and energy/resource consumption.

## 17. Export experiment artifacts

Packages the Module 6e output directory outside itself so the ZIP does not include itself recursively. Run after training and evaluation have completed.

### Code step 184: 17. Export experiment artifacts

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from pathlib import Path

EXPORT_ZIP_PATH = Path("/kaggle/working/ImagePlusRasterTransformerCaptioner.zip")
SOURCE_DIR = Path("/kaggle/working/ERIC_Module6e")

### Code step 185: 17. Export experiment artifacts

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
assert SOURCE_DIR.exists(), f"Output directory does not exist: {SOURCE_DIR}"

### Code step 186: Execute 17. export experiment artifacts

This cell performs one logical operation in the workflow. Review the output before proceeding.

In [ ]:
with ZipFile(EXPORT_ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for file_path in SOURCE_DIR.rglob("*"):
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.relative_to(SOURCE_DIR.parent))

### Code step 187: 17. Export experiment artifacts

This cell contains a small setup, validation, or execution step for the current section.

In [ ]:
print("Exported:", EXPORT_ZIP_PATH)
print(f"ZIP size: {EXPORT_ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

## Reproducibility note

This notebook preserves the uploaded notebook's data preparation, architecture, training, checkpoint loading, caption generation, and qualitative inspection code, and divides evaluation into smaller cells with theory and purpose documented before each step. It has been structurally validated but should be run end-to-end in the target Kaggle runtime to verify package availability, paths, and dependencies.